In [1]:
from transformers import Wav2Vec2Processor, Wav2Vec2ForCTC
import torch
import torchaudio 
import torchaudio.transforms as T

from datasets import load_dataset

import sys
sys.path.append('/om4/group/mcdermott/user/imgriff/projects/End-to-end-ASR-Pytorch/')
from corpus.single_word import SingleWordDataset as Dataset

import fuzzy 
from tqdm.notebook import tqdm
import pandas as pd

In [2]:

path = '/om2/user/imgriff/projects/cocktail_party/datasets/GigaSpeech_top_200_words/' # Path to raw LibriSpeech dataset
dev_split = ['GigaSpeech_top_200_words']  # Name of data splits to be used as validation set


In [3]:
clean_dataset = Dataset(path, dev_split, None, 1)


Using custom data configuration default-6e977ecf05190e74
Reusing dataset csv (/home/imgriff/.cache/huggingface/datasets/csv/default-6e977ecf05190e74/0.0.0/6b9057d9e23d9d8a2f05b985917a0da84d70c5dae3d22ddd8a3f22fb01c69d9e)


In [4]:
sampling_rate = 16000

In [5]:
processor = Wav2Vec2Processor.from_pretrained("facebook/wav2vec2-base-960h")
model = Wav2Vec2ForCTC.from_pretrained("facebook/wav2vec2-base-960h")

Some weights of Wav2Vec2ForCTC were not initialized from the model checkpoint at facebook/wav2vec2-base-960h and are newly initialized: ['wav2vec2.masked_spec_embed']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


## Check performance on demo dataset

In [7]:
dataset2 = load_dataset("hf-internal-testing/librispeech_asr_demo", "clean", split="validation")
dataset2 = dataset2.sort("id")
sampling_rate = dataset2.features["audio"].sampling_rate

Reusing dataset librispeech_asr (/home/imgriff/.cache/huggingface/datasets/hf-internal-testing___librispeech_asr/clean/2.1.0/d3bc4c2bc2078fcde3ad0f0f635862e4c0fef78ba94c4a34c4c250a097af240b)
Loading cached sorted indices for dataset at /home/imgriff/.cache/huggingface/datasets/hf-internal-testing___librispeech_asr/clean/2.1.0/d3bc4c2bc2078fcde3ad0f0f635862e4c0fef78ba94c4a34c4c250a097af240b/cache-2f7c0cbee6ef3aa1.arrow


In [6]:
model = model.cuda()


In [8]:
# results = []

for ix in range(5):
    true = dataset2[ix]["text"]
    inputs = processor(dataset2[ix]["audio"]['array'], sampling_rate=sampling_rate, return_tensors="pt").to('cuda')
    with torch.no_grad():
        logits = model(**inputs).logits
    predicted_ids = torch.argmax(logits, dim=-1)

    # transcribe speech
    transcription = processor.batch_decode(predicted_ids)
    print(f'guess: {transcription[0]}')
    print(f'truth: {true}')
    print()
    
#     results.append({'hyp': transcription[0],
#                     'truth': true})

guess: MISTER QUILTER IS THE APOSTLE OF THE MIDDLE CLASSES AND WE ARE GLAD TO WELCOME HIS GOSPEL
truth: MISTER QUILTER IS THE APOSTLE OF THE MIDDLE CLASSES AND WE ARE GLAD TO WELCOME HIS GOSPEL

guess: NOR IS MISTER QUILTER'S MANNER LESS INTERESTING THAN HIS MATTER
truth: NOR IS MISTER QUILTER'S MANNER LESS INTERESTING THAN HIS MATTER

guess: HE TELLS US THAT AT THIS FESTIVE SEASON OF THE YEAR WITH CHRISTMANUS AND ROAST BEEF LOOMING BEFORE US SIMILES DRAWN FROM EATING AND ITS RESULTS OCCUR MOST READILY TO THE MIND
truth: HE TELLS US THAT AT THIS FESTIVE SEASON OF THE YEAR WITH CHRISTMAS AND ROAST BEEF LOOMING BEFORE US SIMILES DRAWN FROM EATING AND ITS RESULTS OCCUR MOST READILY TO THE MIND

guess: HE HAS GRAVED DOUBTS WHETHER SIR FREDERICK LAYTON'S WORK IS READY GREEK AFTER ALL AND CAN DISCOVER IN IT BUT LITTLE OF ROCKY ITHACA
truth: HE HAS GRAVE DOUBTS WHETHER SIR FREDERICK LEIGHTON'S WORK IS REALLY GREEK AFTER ALL AND CAN DISCOVER IN IT BUT LITTLE OF ROCKY ITHACA

guess: LINILL'S PI

We can see this is a character model based on the handful of errors made. Overall performance looks good.

## Check on Single Word Corpora

no language model is getting used, so things may be messy 

In [7]:
def run_inference(model, dataset):
    results = []
    
    if model.device.type != 'cuda' and torch.cuda.is_available():
        model = model.cuda()
        
    device = model.device.type
    
    for sample in tqdm(dataset):
        true = sample["word"]
        inputs = processor(sample["audio"], sampling_rate=sampling_rate, return_tensors="pt").to(device)
        with torch.no_grad():
            logits = model(**inputs).logits
        predicted_ids = torch.argmax(logits, dim=-1)

        # transcribe speech
        transcription = processor.batch_decode(predicted_ids)

        results.append({'hyp': transcription[0],
                        'truth': true})
    return results

In [9]:
clean_dataset.dataset

Dataset({
    features: ['Unnamed: 0', 'word', 'wav_path'],
    num_rows: 200
})

In [10]:
# resampler = T.Resample(44100, 16000, lowpass_filter_width=64,
#                    rolloff=0.9475937167399596, 
#                    resampling_method="kaiser_window",beta=14.769656459379492, dtype=torch.float32)

def load_audio(example):
    wav, sr = torchaudio.load(example['wav_path'])
#     wav = resampler(wav)
    example['audio'] = wav
    example['sampling_rate'] = 16000
    return example


In [11]:
updated_dataset = clean_dataset.dataset.map(load_audio)

Loading cached processed dataset at /home/imgriff/.cache/huggingface/datasets/csv/default-6e977ecf05190e74/0.0.0/6b9057d9e23d9d8a2f05b985917a0da84d70c5dae3d22ddd8a3f22fb01c69d9e/cache-c28c661dad041df8.arrow


In [12]:
clean_results = run_inference(model, updated_dataset)

  0%|          | 0/200 [00:00<?, ?it/s]

In [15]:
clean_results = pd.DataFrame(clean_results)
clean_results['condition'] = 'clean'

In [16]:
clean_results.head(5)

,hyp,truth,condition
0,THE,THE,clean
1,AND,AND,clean
2,TWO,TO,clean
3,OF,OF,clean
4,A,A,clean


In [17]:
top_match_guesses = clean_results[(clean_results['hyp'] == clean_results['truth'])]
percent_right = 100 * len(top_match_guesses) / len(clean_results.truth.unique())
print(f"{len(top_match_guesses)} matched guesses; {round(percent_right,2)}% correct")
top_match_guesses.head(5)

174 matched guesses; 87.0% correct


,hyp,truth,condition
0,THE,THE,clean
1,AND,AND,clean
3,OF,OF,clean
4,A,A,clean
5,THAT,THAT,clean


In [19]:
dmeta = fuzzy.DMetaphone()

clean_results['hyp_dmeta'] = clean_results['hyp'].apply(lambda x: dmeta(x))
clean_results['truth_dmeta'] = clean_results['truth'].apply(lambda x: dmeta(x))

In [20]:
top_match_guesses = clean_results[(clean_results['hyp_dmeta'] == clean_results['truth_dmeta'])]
percent_right = 100 * len(top_match_guesses) / len(clean_results.truth.unique())
print(f"{len(top_match_guesses)} matched guesses; {round(percent_right,2)}% correct")
top_match_guesses.head(5)

190 matched guesses; 95.0% correct


,hyp,truth,condition,hyp_dmeta,truth_dmeta
0,THE,THE,clean,"[b'0', b'T']","[b'0', b'T']"
1,AND,AND,clean,"[b'ANT', None]","[b'ANT', None]"
2,TWO,TO,clean,"[b'T', None]","[b'T', None]"
3,OF,OF,clean,"[b'AF', None]","[b'AF', None]"
4,A,A,clean,"[b'A', None]","[b'A', None]"


## On noise 0 dBSNR

In [21]:

path = '/om2/user/imgriff/projects/cocktail_party/datasets/GigaSpeech_top_200_words/' # Path to raw LibriSpeech dataset
dev_split = ['GigaSpeech_top_200_words_0SNR']  # Name of data splits to be used as validation set

low_dataset = Dataset(path, dev_split, None, 1)

updated_dataset = low_dataset.dataset.map(load_audio)

Using custom data configuration default-c9dd0aeaa110b3ae
Reusing dataset csv (/home/imgriff/.cache/huggingface/datasets/csv/default-c9dd0aeaa110b3ae/0.0.0/6b9057d9e23d9d8a2f05b985917a0da84d70c5dae3d22ddd8a3f22fb01c69d9e)
Loading cached processed dataset at /home/imgriff/.cache/huggingface/datasets/csv/default-c9dd0aeaa110b3ae/0.0.0/6b9057d9e23d9d8a2f05b985917a0da84d70c5dae3d22ddd8a3f22fb01c69d9e/cache-1aa1c950d7d80a14.arrow


In [22]:
low_results = run_inference(model, updated_dataset)

  0%|          | 0/200 [00:00<?, ?it/s]

In [23]:

low_results = pd.DataFrame(low_results)
low_results['condition'] = '0dB SNR'
top_match_guesses = low_results[(low_results['hyp'] == low_results['truth'])]
percent_right = 100 * len(top_match_guesses) / len(low_results.truth.unique())
print(f"{len(top_match_guesses)} matched guesses; {round(percent_right,2)}% correct")
top_match_guesses.head(5)

33 matched guesses; 16.5% correct


,hyp,truth,condition
7,I,I,0dB SNR
10,IT,IT,0dB SNR
14,THIS,THIS,0dB SNR
26,AT,AT,0dB SNR
40,THERE,THERE,0dB SNR


In [24]:

low_results['hyp_dmeta'] = low_results['hyp'].apply(lambda x: dmeta(x) if isinstance(x, str) else x)
low_results['truth_dmeta'] = low_results['truth'].apply(lambda x: dmeta(x))

top_match_guesses = low_results[(low_results['hyp_dmeta'] == low_results['truth_dmeta'])]
percent_right = 100 * len(top_match_guesses) / len(low_results.truth.unique())
print(f"{len(top_match_guesses)} matched guesses; {round(percent_right,2)}% correct")
top_match_guesses.head(5)

50 matched guesses; 25.0% correct


,hyp,truth,condition,hyp_dmeta,truth_dmeta
2,TOO,TO,0dB SNR,"[b'T', None]","[b'T', None]"
4,E,A,0dB SNR,"[b'A', None]","[b'A', None]"
7,I,I,0dB SNR,"[b'A', None]","[b'A', None]"
10,IT,IT,0dB SNR,"[b'AT', None]","[b'AT', None]"
13,SI,SO,0dB SNR,"[b'S', None]","[b'S', None]"


## On noise -5 dBSNR

In [25]:

path = '/om2/user/imgriff/projects/cocktail_party/datasets/GigaSpeech_top_200_words/' # Path to raw LibriSpeech dataset
dev_split = ['GigaSpeech_top_200_words_-5SNR']  # Name of data splits to be used as validation set

high_dataset = Dataset(path, dev_split, None, 1)

updated_dataset = high_dataset.dataset.map(load_audio)

Using custom data configuration default-5683877c4c4495f8
Reusing dataset csv (/home/imgriff/.cache/huggingface/datasets/csv/default-5683877c4c4495f8/0.0.0/6b9057d9e23d9d8a2f05b985917a0da84d70c5dae3d22ddd8a3f22fb01c69d9e)
Loading cached processed dataset at /home/imgriff/.cache/huggingface/datasets/csv/default-5683877c4c4495f8/0.0.0/6b9057d9e23d9d8a2f05b985917a0da84d70c5dae3d22ddd8a3f22fb01c69d9e/cache-c3c5b179e5247f84.arrow


In [26]:
high_results = run_inference(model, updated_dataset)

  0%|          | 0/200 [00:00<?, ?it/s]

In [28]:

high_results = pd.DataFrame(high_results)
high_results['condition'] = '-5dB'
top_match_guesses = high_results[(high_results['hyp'] == high_results['truth'])]
percent_right = 100 * len(top_match_guesses) / len(high_results.truth.unique())
print(f"{len(top_match_guesses)} matched guesses; {round(percent_right,2)}% correct")
top_match_guesses.head(5)

1 matched guesses; 0.5% correct


,hyp,truth
41,JUST,JUST


In [30]:

high_results['hyp_dmeta'] = high_results['hyp'].apply(lambda x: dmeta(x) if isinstance(x, str) else x)
high_results['truth_dmeta'] = high_results['truth'].apply(lambda x: dmeta(x))

top_match_guesses = high_results[(high_results['hyp_dmeta'] == high_results['truth_dmeta'])]
percent_right = 100 * len(top_match_guesses) / len(high_results.truth.unique())
print(f"{len(top_match_guesses)} matched guesses; {round(percent_right,2)}% correct")
top_match_guesses.head(5)

7 matched guesses; 3.5% correct


,hyp,truth,hyp_dmeta,truth_dmeta
13,SY,SO,"[b'S', None]","[b'S', None]"
38,DE,DO,"[b'T', None]","[b'T', None]"
41,JUST,JUST,"[b'JST', b'AST']","[b'JST', b'AST']"
87,SE,SEE,"[b'S', None]","[b'S', None]"
91,WA,WAY,"[b'A', b'F']","[b'A', b'F']"


In [33]:
all_results = pd.concat([clean_results, low_results, high_results])

In [34]:
all_results

,hyp,truth,condition,hyp_dmeta,truth_dmeta
0,THE,THE,clean,"[b'0', b'T']","[b'0', b'T']"
1,AND,AND,clean,"[b'ANT', None]","[b'ANT', None]"
2,TWO,TO,clean,"[b'T', None]","[b'T', None]"
3,OF,OF,clean,"[b'AF', None]","[b'AF', None]"
4,A,A,clean,"[b'A', None]","[b'A', None]"
...,...,...,...,...,...
195,,TWENTY,NaN,"[None, None]","[b'TNT', None]"
196,,EACH,NaN,"[None, None]","[b'AK', None]"
197,,SURE,NaN,"[None, None]","[b'SR', None]"
198,,LET,NaN,"[None, None]","[b'LT', None]"
